# JSWidget Demo - Custom JavaScript Rendering Widget

This notebook demonstrates the `JSWidget` - a lightweight widget that:
1. Accepts and renders arbitrary JavaScript (Canvas, WebGL, etc.)
2. Passes binary data (e.g. mesh buffers) via base64-encoded traitlets
3. Works in VS Code, JupyterLab, and classic notebooks without extra dependencies

In [ ]:
from jswidget import JSWidget
import numpy as np

## Example 1: Simple Canvas Drawing

In [ ]:
w = JSWidget(width=600, height=400)
w.js_code = '''
    const canvas = document.createElement("canvas");
    canvas.width = opts.width;
    canvas.height = opts.height;
    canvas.style.border = "1px solid #ccc";
    el.appendChild(canvas);
    const ctx = canvas.getContext("2d");

    // Draw gradient background
    const grad = ctx.createLinearGradient(0, 0, canvas.width, canvas.height);
    grad.addColorStop(0, "#1a1a2e");
    grad.addColorStop(1, "#16213e");
    ctx.fillStyle = grad;
    ctx.fillRect(0, 0, canvas.width, canvas.height);

    // Draw text
    ctx.fillStyle = "#e94560";
    ctx.font = "24px sans-serif";
    ctx.fillText("JSWidget - No anywidget dependency", 20, 50);
    ctx.fillStyle = "#0f3460";
    ctx.font = "16px sans-serif";
    ctx.fillText("Rendering JavaScript via IPython.display.HTML", 20, 80);
'''
w.show()

## Example 2: Passing Data via Traitlets

In [ ]:
w2 = JSWidget(width=600, height=300)
w2.data = {'values': [10, 40, 80, 30, 60, 90, 45, 70], 'color': '#4ecdc4'}
w2.js_code = '''
    const canvas = document.createElement("canvas");
    canvas.width = opts.width;
    canvas.height = opts.height;
    canvas.style.border = "1px solid #ccc";
    el.appendChild(canvas);
    const ctx = canvas.getContext("2d");

    function draw(d) {
        const values = d.values || [];
        const color = d.color || "#4ecdc4";
        const w = canvas.width, h = canvas.height;

        ctx.clearRect(0, 0, w, h);
        ctx.fillStyle = "#1a1a2e";
        ctx.fillRect(0, 0, w, h);

        if (values.length === 0) return;
        const barW = (w - 40) / values.length;
        const maxVal = Math.max(...values);

        values.forEach((v, i) => {
            const barH = (v / maxVal) * (h - 60);
            ctx.fillStyle = color;
            ctx.fillRect(20 + i * barW + 2, h - 30 - barH, barW - 4, barH);
        });
    }

    draw(data);
    // Re-draw when data is updated from Python
    onData(draw);
'''
w2.show()

In [ ]:
# Update data from Python - triggers onData callbacks in JS
w2.send_data({'values': [90, 20, 55, 75, 35, 10, 85, 50], 'color': '#ff6b6b'})

## Example 3: WebGL Mesh Rendering with Binary Buffers

This demonstrates passing mesh data (vertices, normals, indices) as binary buffers over the comm API.

In [ ]:
# Generate a simple sphere mesh
def make_sphere(radius=1.0, slices=32, stacks=16):
    vertices = []
    normals = []
    indices = []

    for i in range(stacks + 1):
        phi = np.pi * i / stacks
        for j in range(slices + 1):
            theta = 2.0 * np.pi * j / slices
            x = radius * np.sin(phi) * np.cos(theta)
            y = radius * np.cos(phi)
            z = radius * np.sin(phi) * np.sin(theta)
            vertices.append([x, y, z])
            normals.append([x/radius, y/radius, z/radius])

    for i in range(stacks):
        for j in range(slices):
            a = i * (slices + 1) + j
            b = a + slices + 1
            indices.extend([a, b, a + 1, b, b + 1, a + 1])

    return (np.array(vertices, dtype=np.float32),
            np.array(normals, dtype=np.float32),
            np.array(indices, dtype=np.uint32))

vertices, normals, indices = make_sphere()
print(f"Mesh: {len(vertices)} vertices, {len(indices)//3} triangles")

In [ ]:
w3 = JSWidget(width=700, height=500)
w3.data = {'opacity': 1.0, 'color': [0.9, 0.2, 0.2]}
w3.set_buffers(vertices=vertices, normals=normals, indices=indices)
w3.js_code = '''
    const canvas = document.createElement("canvas");
    canvas.width = opts.width;
    canvas.height = opts.height;
    canvas.style.border = "1px solid #333";
    canvas.tabIndex = 0;
    el.appendChild(canvas);

    const gl = canvas.getContext("webgl2");
    if (!gl) { el.textContent = "WebGL2 not supported"; return; }

    // Shaders
    const vsSource = `#version 300 es
        uniform mat4 uMVP;
        uniform mat4 uNormalMatrix;
        in vec3 aPosition;
        in vec3 aNormal;
        out vec3 vNormal;
        out vec3 vPos;
        void main() {
            vNormal = (uNormalMatrix * vec4(aNormal, 0.0)).xyz;
            vPos = aPosition;
            gl_Position = uMVP * vec4(aPosition, 1.0);
        }`;
    const fsSource = `#version 300 es
        precision highp float;
        uniform float uOpacity;
        uniform vec3 uColor;
        in vec3 vNormal;
        in vec3 vPos;
        out vec4 fragColor;
        void main() {
            vec3 n = normalize(vNormal);
            vec3 light = normalize(vec3(0.5, 1.0, 0.8));
            float diff = max(dot(n, light), 0.0);
            float amb = 0.2;
            vec3 col = uColor * (amb + 0.8 * diff);
            fragColor = vec4(col, uOpacity);
        }`;

    function compileShader(src, type) {
        const s = gl.createShader(type);
        gl.shaderSource(s, src);
        gl.compileShader(s);
        if (!gl.getShaderParameter(s, gl.COMPILE_STATUS))
            console.error(gl.getShaderInfoLog(s));
        return s;
    }
    const prog = gl.createProgram();
    gl.attachShader(prog, compileShader(vsSource, gl.VERTEX_SHADER));
    gl.attachShader(prog, compileShader(fsSource, gl.FRAGMENT_SHADER));
    gl.linkProgram(prog);
    gl.useProgram(prog);

    const locs = {
        aPosition: gl.getAttribLocation(prog, "aPosition"),
        aNormal: gl.getAttribLocation(prog, "aNormal"),
        uMVP: gl.getUniformLocation(prog, "uMVP"),
        uNormalMatrix: gl.getUniformLocation(prog, "uNormalMatrix"),
        uOpacity: gl.getUniformLocation(prog, "uOpacity"),
        uColor: gl.getUniformLocation(prog, "uColor"),
    };

    // Upload mesh from buffers
    let vao = null, indexCount = 0;
    function uploadMesh() {
        const vertBuf = getBuffer("vertices");
        const normBuf = getBuffer("normals");
        const idxBuf = getBuffer("indices");
        if (!vertBuf || !normBuf || !idxBuf) return;

        if (vao) gl.deleteVertexArray(vao);
        const verts = new Float32Array(vertBuf);
        const norms = new Float32Array(normBuf);
        const idx = new Uint32Array(idxBuf);
        indexCount = idx.length;

        vao = gl.createVertexArray();
        gl.bindVertexArray(vao);

        const pb = gl.createBuffer();
        gl.bindBuffer(gl.ARRAY_BUFFER, pb);
        gl.bufferData(gl.ARRAY_BUFFER, verts, gl.STATIC_DRAW);
        gl.enableVertexAttribArray(locs.aPosition);
        gl.vertexAttribPointer(locs.aPosition, 3, gl.FLOAT, false, 0, 0);

        const nb = gl.createBuffer();
        gl.bindBuffer(gl.ARRAY_BUFFER, nb);
        gl.bufferData(gl.ARRAY_BUFFER, norms, gl.STATIC_DRAW);
        gl.enableVertexAttribArray(locs.aNormal);
        gl.vertexAttribPointer(locs.aNormal, 3, gl.FLOAT, false, 0, 0);

        const ib = gl.createBuffer();
        gl.bindBuffer(gl.ELEMENT_ARRAY_BUFFER, ib);
        gl.bufferData(gl.ELEMENT_ARRAY_BUFFER, idx, gl.STATIC_DRAW);

        gl.bindVertexArray(null);
        render();
    }

    // Camera state (preserved across re-renders)
    const prev = getState();
    let rotX = prev.rotX || 0.3, rotY = prev.rotY || 0.5, zoom = prev.zoom || 3.0;
    let dragging = false, lastX = 0, lastY = 0;

    // Column-major 4x4 matrix utilities
    function mat4Perspective(fov, aspect, near, far) {
        const f = 1.0 / Math.tan(fov / 2);
        const nf = 1 / (near - far);
        return [f/aspect,0,0,0, 0,f,0,0, 0,0,(far+near)*nf,-1, 0,0,2*far*near*nf,0];
    }
    function mat4Multiply(a, b) {
        const r = new Float32Array(16);
        for (let col = 0; col < 4; col++)
            for (let row = 0; row < 4; row++) {
                let s = 0;
                for (let k = 0; k < 4; k++) s += a[k*4+row] * b[col*4+k];
                r[col*4+row] = s;
            }
        return r;
    }
    function mat4RotX(a) {
        const c=Math.cos(a),s=Math.sin(a);
        return [1,0,0,0, 0,c,s,0, 0,-s,c,0, 0,0,0,1];
    }
    function mat4RotY(a) {
        const c=Math.cos(a),s=Math.sin(a);
        return [c,0,-s,0, 0,1,0,0, s,0,c,0, 0,0,0,1];
    }
    function mat4Translate(x,y,z) {
        return [1,0,0,0, 0,1,0,0, 0,0,1,0, x,y,z,1];
    }

    let currentData = data;
    function render() {
        if (!vao) return;
        const opacity = currentData.opacity !== undefined ? currentData.opacity : 1.0;
        const color = currentData.color || [0.9, 0.2, 0.2];

        gl.viewport(0, 0, canvas.width, canvas.height);
        gl.clearColor(0.1, 0.1, 0.15, 1.0);
        gl.clear(gl.COLOR_BUFFER_BIT | gl.DEPTH_BUFFER_BIT);
        gl.enable(gl.DEPTH_TEST);

        const aspect = canvas.width / canvas.height;
        const proj = mat4Perspective(Math.PI/4, aspect, 0.1, 100.0);
        const view = mat4Translate(0, 0, -zoom);
        const rx = mat4RotX(rotX);
        const ry = mat4RotY(rotY);
        const modelMat = mat4Multiply(rx, ry);
        const mv = mat4Multiply(view, modelMat);
        const mvp = mat4Multiply(proj, mv);

        gl.useProgram(prog);
        gl.uniformMatrix4fv(locs.uMVP, false, mvp);
        gl.uniformMatrix4fv(locs.uNormalMatrix, false, modelMat);
        gl.uniform1f(locs.uOpacity, opacity);
        gl.uniform3fv(locs.uColor, color);

        gl.bindVertexArray(vao);
        gl.drawElements(gl.TRIANGLES, indexCount, gl.UNSIGNED_INT, 0);
        gl.bindVertexArray(null);
    }

    // Mouse interaction
    canvas.addEventListener("mousedown", e => {
        dragging = true; lastX = e.offsetX; lastY = e.offsetY;
        canvas.focus(); e.preventDefault();
    });
    canvas.addEventListener("mousemove", e => {
        if (!dragging) return;
        rotY += (e.offsetX - lastX) * 0.01;
        rotX += (e.offsetY - lastY) * 0.01;
        lastX = e.offsetX; lastY = e.offsetY;
        setState({rotX, rotY, zoom});
        render();
    });
    canvas.addEventListener("mouseup", () => { dragging = false; });
    canvas.addEventListener("wheel", e => {
        zoom *= 1 + e.deltaY * 0.001;
        zoom = Math.max(1, Math.min(20, zoom));
        setState({rotX, rotY, zoom});
        render(); e.preventDefault();
    }, {passive: false});
    canvas.addEventListener("contextmenu", e => e.preventDefault());

    // Handle live data updates from Python
    onData((newData) => {
        currentData = newData;
        render();
    });

    // Handle live buffer updates from Python
    onBuffers(() => {
        uploadMesh();
    });

    // Initial mesh upload
    uploadMesh();
'''
w3.show()

In [ ]:
# Update rendering parameters from Python
w3.send_data({'opacity': 0.85, 'color': [0.2, 0.7, 0.9]})

In [ ]:
# Update mesh dynamically - swap to a torus
def make_torus(R=1.0, r=0.3, slices=32, stacks=16):
    vertices, normals, indices = [], [], []
    for i in range(stacks + 1):
        phi = 2.0 * np.pi * i / stacks
        for j in range(slices + 1):
            theta = 2.0 * np.pi * j / slices
            x = (R + r * np.cos(theta)) * np.cos(phi)
            y = r * np.sin(theta)
            z = (R + r * np.cos(theta)) * np.sin(phi)
            nx = np.cos(theta) * np.cos(phi)
            ny = np.sin(theta)
            nz = np.cos(theta) * np.sin(phi)
            vertices.append([x, y, z])
            normals.append([nx, ny, nz])
    for i in range(stacks):
        for j in range(slices):
            a = i * (slices + 1) + j
            b = a + slices + 1
            indices.extend([a, b, a + 1, b, b + 1, a + 1])
    return (np.array(vertices, dtype=np.float32),
            np.array(normals, dtype=np.float32),
            np.array(indices, dtype=np.uint32))

tv, tn, ti = make_torus()
w3.set_buffers(vertices=tv, normals=tn, indices=ti)
w3.send_data({'opacity': 1.0, 'color': [0.9, 0.6, 0.1]})
print(f"Updated to torus: {len(tv)} vertices, {len(ti)//3} triangles")